# Notebook 2: Condicionales y Bifurcaciones en LangGraph

En el notebook anterior todos los grafos eran lineales: A → B → C.
En la realidad los agentes necesitan **tomar decisiones**: dependiendo del input, ejecutan un camino u otro.

En este notebook aprenderás:
- Qué son los **conditional edges** y cómo funcionan
- Cómo escribir una **función de routing** (enrutamiento)
- Cómo usar `add_conditional_edges` para bifurcar el flujo
- Cómo construir un **clasificador de intención** que desvía el flujo según el tipo de pregunta

## 1. Instalación y configuración

In [ ]:
# %pip install -qU langgraph langchain-google-genai

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.3,  # Más determinista para clasificación
)
print("Modelo listo:", llm.model)

## 2. El problema: diferentes preguntas necesitan diferentes respuestas

Imaginemos un asistente que recibe preguntas de tres tipos:
- **Matemáticas**: necesita un nodo especializado que razone paso a paso
- **Cultura general**: respuesta directa con el LLM
- **Código**: responde con un ejemplo de código

El flujo que vamos a construir:

```
                  ┌─── nodo_matematicas ───┐
START → clasificar─┼─── nodo_cultura      ──┼→ END
                  └─── nodo_codigo       ───┘
```

## 3. Definir el Estado

In [ ]:
from typing import TypedDict, Literal

class EstadoAsistente(TypedDict):
    pregunta: str
    tipo: str           # 'matematicas', 'cultura', 'codigo'
    respuesta: str
    nodo_usado: str     # Para saber qué rama tomó el grafo

## 4. Nodo clasificador

Este nodo llama al LLM para que clasifique la pregunta y actualiza el campo `tipo` del estado.

In [ ]:
def nodo_clasificar(estado: EstadoAsistente) -> dict:
    """
    Clasifica la pregunta del usuario en una de tres categorías.
    """
    prompt = f"""Clasifica la siguiente pregunta en UNA de estas categorías:
- matematicas (si involucra cálculos, álgebra, estadística, lógica numérica)
- codigo (si pide escribir, debuggear o explicar código)
- cultura (cualquier otra pregunta de conocimiento general)

Responde SOLO con una de estas palabras: matematicas, codigo, cultura

Pregunta: {estado['pregunta']}"""

    respuesta = llm.invoke(prompt)
    tipo = respuesta.content.strip().lower()
    
    # Normalizar en caso de respuesta inesperada
    if tipo not in ["matematicas", "codigo", "cultura"]:
        tipo = "cultura"
    
    print(f"[Clasificador] Pregunta: '{estado['pregunta'][:50]}' → Tipo: '{tipo}'")
    return {"tipo": tipo}

## 5. Nodos especializados (las ramas)

In [ ]:
def nodo_matematicas(estado: EstadoAsistente) -> dict:
    """
    Resuelve preguntas matemáticas paso a paso.
    """
    prompt = f"""Eres un tutor de matemáticas. Resuelve la siguiente pregunta mostrando
cada paso claramente. Usa notación clara.

Pregunta: {estado['pregunta']}"""
    
    respuesta = llm.invoke(prompt)
    print("[Nodo Matemáticas] Procesando...")
    return {
        "respuesta": respuesta.content,
        "nodo_usado": "matematicas"
    }


def nodo_cultura(estado: EstadoAsistente) -> dict:
    """
    Responde preguntas de cultura general de forma concisa.
    """
    prompt = f"""Eres un asistente enciclopédico. Responde de forma concisa y clara.

Pregunta: {estado['pregunta']}"""
    
    respuesta = llm.invoke(prompt)
    print("[Nodo Cultura] Procesando...")
    return {
        "respuesta": respuesta.content,
        "nodo_usado": "cultura"
    }


def nodo_codigo(estado: EstadoAsistente) -> dict:
    """
    Ayuda con preguntas de programación, siempre con ejemplo de código.
    """
    prompt = f"""Eres un experto en programación. Responde siempre incluyendo un ejemplo
de código funcional. Usa bloques de código con el lenguaje apropiado.

Pregunta: {estado['pregunta']}"""
    
    respuesta = llm.invoke(prompt)
    print("[Nodo Código] Procesando...")
    return {
        "respuesta": respuesta.content,
        "nodo_usado": "codigo"
    }

## 6. La función de routing (enrutamiento)

Esta es la clave de los conditional edges. Es una función que:
- Recibe el estado actual
- Retorna un **string** con el nombre del siguiente nodo a ejecutar

LangGraph usará este string para saber a qué nodo ir.

In [ ]:
def router(estado: EstadoAsistente) -> Literal["matematicas", "cultura", "codigo"]:
    """
    Función de routing: lee el tipo del estado y retorna el nombre del nodo destino.
    El valor retornado debe corresponder a un nombre de nodo registrado en el grafo.
    """
    tipo = estado["tipo"]
    print(f"[Router] Dirigiendo al nodo: '{tipo}'")
    return tipo  # 'matematicas', 'cultura' o 'codigo'

## 7. Construir el grafo con conditional edges

In [ ]:
from langgraph.graph import StateGraph, START, END

grafo = StateGraph(EstadoAsistente)

# Agregar todos los nodos
grafo.add_node("clasificar",   nodo_clasificar)
grafo.add_node("matematicas",  nodo_matematicas)
grafo.add_node("cultura",      nodo_cultura)
grafo.add_node("codigo",       nodo_codigo)

# Edge fijo: siempre empieza en 'clasificar'
grafo.add_edge(START, "clasificar")

# Edge condicional: DESPUÉS de 'clasificar', el router decide a dónde ir
grafo.add_conditional_edges(
    source="clasificar",     # Nodo de origen
    path=router,             # Función que decide el destino
    path_map={               # Mapa de valores posibles → nodos destino
        "matematicas": "matematicas",
        "cultura":     "cultura",
        "codigo":      "codigo",
    }
)

# Todos los nodos especializados van al END
grafo.add_edge("matematicas", END)
grafo.add_edge("cultura",     END)
grafo.add_edge("codigo",      END)

app = grafo.compile()
print("✅ Grafo con bifurcaciones compilado")

In [ ]:
# Visualizar el grafo (notarás que ahora tiene múltiples caminos)
print(app.get_graph().draw_ascii())

## 8. Probar el grafo con diferentes preguntas

In [ ]:
def ejecutar_asistente(pregunta: str):
    """Helper para ejecutar el asistente y mostrar el resultado."""
    print("\n" + "=" * 65)
    print(f"PREGUNTA: {pregunta}")
    print("=" * 65)
    
    estado_inicial = {
        "pregunta": pregunta,
        "tipo": "",
        "respuesta": "",
        "nodo_usado": ""
    }
    
    resultado = app.invoke(estado_inicial)
    
    print(f"\n✅ Nodo usado: '{resultado['nodo_usado']}'")
    print(f"\nRESPUESTA:\n{resultado['respuesta']}")
    return resultado

In [ ]:
# Prueba 1: Pregunta matemática
r1 = ejecutar_asistente("¿Cuánto es la raíz cuadrada de 144 más el 15% de 200?")

In [ ]:
# Prueba 2: Pregunta de cultura
r2 = ejecutar_asistente("¿En qué año cayó el muro de Berlín?")

In [ ]:
# Prueba 3: Pregunta de código
r3 = ejecutar_asistente("¿Cómo se hace un decorador en Python?")

## 9. Variante: routing sin path_map

Si la función de routing retorna el nombre exacto del nodo, puedes omitir `path_map`.
LangGraph usará el valor retornado directamente como nombre de nodo.

In [ ]:
# Mismo resultado, sintaxis más corta (cuando los valores son exactamente los nombres de nodo)
grafo2 = StateGraph(EstadoAsistente)
grafo2.add_node("clasificar",  nodo_clasificar)
grafo2.add_node("matematicas", nodo_matematicas)
grafo2.add_node("cultura",     nodo_cultura)
grafo2.add_node("codigo",      nodo_codigo)
grafo2.add_edge(START, "clasificar")

# Sin path_map: la función retorna el nombre del nodo directamente
grafo2.add_conditional_edges("clasificar", router)

grafo2.add_edge("matematicas", END)
grafo2.add_edge("cultura", END)
grafo2.add_edge("codigo", END)

app2 = grafo2.compile()
print("✅ Grafo alternativo (sin path_map) compilado")

## 10. Routing hacia END directamente

Un router también puede enviar el flujo directamente al `END` sin pasar por otro nodo.

In [ ]:
from langgraph.graph import END

# Ejemplo: si la pregunta está vacía, terminar sin procesar
def router_con_guard(estado: EstadoAsistente) -> str:
    if not estado["pregunta"].strip():
        print("[Router] Pregunta vacía, terminando.")
        return END  # Se puede retornar END directamente
    return estado["tipo"]

print("Router con guard definido")

## 11. Ejercicios propuestos

1. **Añade una cuarta categoría** `historia` con un nodo especializado que siempre responda dando el contexto histórico completo.

2. **Dos clasificaciones en cascada**: primero clasifica si la pregunta es "técnica" o "no técnica", y si es técnica, clasifica entre "matemáticas" y "código".

3. **Nodo de fallback**: añade un nodo `desconocido` al que vaya el router cuando la pregunta no encaja en ninguna categoría conocida.

## Resumen

| Concepto | Descripción |
|---|---|
| `add_conditional_edges` | Conecta un nodo a múltiples destinos posibles |
| Función de routing | Recibe el estado, retorna el nombre del siguiente nodo |
| `path_map` | Diccionario que mapea valores retornados a nombres de nodo |
| Retornar `END` | El router puede terminar el grafo directamente |
| Branches independientes | Cada rama puede tener su propia lógica especializada |